# Gaussian elimination with partial pivoting — PyTorch / Kaggle

Create a notebook at https://www.kaggle.com/code and use File → Import Notebook to upload this `.ipynb`. In Settings / Session options, choose an available GPU under Accelerator. Enable Internet for automatic matrix download, or attach a dataset containing your `.mat` files using Add Input. Run the cells in order. PyTorch, NumPy and SciPy are included in Kaggle's Python environment; no repository imports are needed.

The elimination and backward substitution are implemented below. `torch.linalg.solve` is used only in the optional reference benchmark. All computation uses float64. The input A and b are preserved; LU stores U on/above the diagonal and the elimination multipliers below it. One RHS vector is supported. This notebook uses one GPU even if the session provides two.

In [9]:
import time
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import torch
from scipy.io import loadmat
from scipy.sparse import issparse

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU")

PyTorch: 2.11.0+cu130
Device: Tesla T4


In [10]:
@torch.no_grad()
def gaussian_with_partial_pivoting(A, b):
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A must be a nonempty square matrix")
    if b.ndim != 1 or b.shape[0] != A.shape[0]:
        raise ValueError("b must be a vector matching A")
    if A.dtype != torch.float64 or b.dtype != torch.float64 or A.device != b.device:
        raise ValueError("A and b must be float64 tensors on the same device")
    if not (torch.isfinite(A).all() and torch.isfinite(b).all()):
        raise ValueError("A and b must be finite")

    LU, c = A.clone().contiguous(), b.clone().contiguous()
    n = len(c)
    for k in range(n):
        r = k + torch.argmax(LU[k:, k].abs()).item()
        if r != k:
            LU[[k, r], :] = LU[[r, k], :]
            c[[k, r]] = c[[r, k]]
        pivot = LU[k, k]
        if not torch.isfinite(pivot).item() or pivot.item() == 0.0:
            raise ValueError(f"Zero or nonfinite pivot at column {k}")
        if k + 1 < n:
            LU[k+1:, k].div_(pivot)
            multipliers = LU[k+1:, k].clone()
            pivot_row = LU[k, k+1:].clone()
            LU[k+1:, k+1:].addr_(multipliers, pivot_row, beta=1, alpha=-1)
            c[k+1:].add_(multipliers * c[k], alpha=-1)
    return LU, c


@torch.no_grad()
def backward_substitution(LU, c):
    x = torch.empty_like(c)
    for i in range(len(c) - 1, -1, -1):
        x[i] = (c[i] - torch.dot(LU[i, i+1:], x[i+1:])) / LU[i, i]
    return x


def solve_ge(A, b):
    LU, c = gaussian_with_partial_pivoting(A, b)
    return backward_substitution(LU, c)


def timed_solve(fn, A, b):
    if A.is_cuda:
        torch.cuda.synchronize(A.device)
    start = time.perf_counter()
    x = fn(A, b)
    if A.is_cuda:
        torch.cuda.synchronize(A.device)
    return x, time.perf_counter() - start


def report(label, A, b, x, x_true, seconds):
    residual = (torch.linalg.vector_norm(A @ x - b) / torch.linalg.vector_norm(b)).item()
    error = (torch.linalg.vector_norm(x - x_true) / torch.linalg.vector_norm(x_true)).item()
    print(f"{label}: {seconds:.6f} s | relative residual {residual:.3e} | relative solution error {error:.3e}")

## Small checks

The repository example has solution `[1, 1, 2]`. The second system forces row swaps at multiple elimination steps. A small warm-up initializes the kernels before timing the SuiteSparse solve.

In [11]:
A_small = torch.tensor([[2, 1, 1], [4, 3, 3], [8, 7, 9]], dtype=dtype, device=device)
b_small = torch.tensor([5, 13, 33], dtype=dtype, device=device)
x_small = solve_ge(A_small, b_small)
torch.testing.assert_close(x_small, torch.tensor([1, 1, 2], dtype=dtype, device=device))
print("Repository example:", x_small.cpu().numpy())

A_swap = torch.tensor([[0, 2, 1], [1, 0, 3], [4, 5, 6]], dtype=dtype, device=device)
x_expected = torch.tensor([1, -2, 3], dtype=dtype, device=device)
b_swap = A_swap @ x_expected
A_before, b_before = A_swap.clone(), b_swap.clone()
torch.testing.assert_close(solve_ge(A_swap, b_swap), x_expected, rtol=1e-12, atol=1e-12)
assert torch.equal(A_swap, A_before) and torch.equal(b_swap, b_before)
print("Row-swap and input-preservation checks passed")

_ = solve_ge(A_small, b_small)
_ = torch.linalg.solve(A_small, b_small)
if device.type == "cuda":
    torch.cuda.synchronize(device)

Repository example: [1. 1. 2.]
Row-swap and input-preservation checks passed


## Obtain the SuiteSparse MATLAB file

Default: HB/bcsstk17. To try the smaller matrix first, set `MATRIX = "bcsstk09"`.

The next cell first searches attached Kaggle datasets under `/kaggle/input` for the selected filename. Upload your repository's `data/bcsstk17.mat` and optionally `data/bcsstk09.mat` as a private Kaggle dataset, then attach it using Add Input. You can also set `MAT_PATH` to a specific attached file.

If no attached file is found, the cell downloads the matrix into `/kaggle/working`. Enable Internet in Session options for this route. If the download service fails, use the attached dataset route instead.

In [12]:
MATRIX = "bcsstk17"  # "bcsstk09" for the smaller test
MAT_PATH = None  # Optional: "/kaggle/input/your-dataset/bcsstk17.mat"

if MATRIX not in {"bcsstk17", "bcsstk09"}:
    raise ValueError("Choose bcsstk17 or bcsstk09")

if MAT_PATH is not None:
    path = Path(MAT_PATH)
    if not path.is_file():
        raise FileNotFoundError(path)
else:
    input_dir = Path("/kaggle/input")
    matches = sorted(input_dir.rglob(f"{MATRIX}.mat")) if input_dir.exists() else []
    if len(matches) > 1:
        raise ValueError(f"Multiple matching files: {matches}. Set MAT_PATH explicitly.")
    if matches:
        path = matches[0]
    else:
        work_dir = Path("/kaggle/working")
        if not work_dir.exists():
            work_dir = Path.cwd()
        path = work_dir / f"{MATRIX}.mat"
        if not path.exists():
            url = f"https://sparse.tamu.edu/mat/HB/{MATRIX}.mat"
            try:
                urlretrieve(url, path)
            except Exception as exc:
                path.unlink(missing_ok=True)
                raise RuntimeError(
                    "Download failed. Enable Internet in Kaggle Session options, "
                    "or attach a dataset containing the .mat file using Add Input."
                ) from exc
print("File:", path)

File: /kaggle/input/datasets/rajathrajesh12/bcsstk17/bcsstk17.mat


In [13]:
mat = loadmat(path)
A_sparse = mat["Problem"][0, 0]["A"]
if A_sparse.shape[0] != A_sparse.shape[1]:
    raise ValueError("Expected a square matrix")
print("Shape:", A_sparse.shape)
print(f"Each dense float64 matrix: {A_sparse.shape[0]**2 * 8 / 2**30:.2f} GiB")
A_numpy = A_sparse.toarray() if issparse(A_sparse) else np.asarray(A_sparse)
A = torch.tensor(A_numpy, dtype=dtype, device=device)
del A_numpy, A_sparse, mat
x_true = torch.ones(A.shape[0], dtype=dtype, device=device)
b = A @ x_true
print("dtype:", A.dtype, "device:", A.device)

Shape: (10974, 10974)
Each dense float64 matrix: 0.90 GiB
dtype: torch.float64 device: cuda:0


## Solve and measure

The timer includes input cloning, validation, pivot selection, elimination and backward substitution. Downloads, loading, host-to-device transfers, RHS construction and error calculations are excluded. GPU timing synchronizes before and after the solve, so queued kernels are included. These are single-run wall times after a small warm-up, not a rigorous performance study.

In [14]:
x, seconds = timed_solve(solve_ge, A, b)
report("Gaussian elimination", A, b, x, x_true, seconds)

Gaussian elimination: 35.474346 s | relative residual 2.391e-15 | relative solution error 9.679e-13


In [17]:
RUN_REFERENCE = True

if RUN_REFERENCE:
    x_reference, reference_seconds = timed_solve(torch.linalg.solve, A, b)
    report("torch.linalg.solve reference", A, b, x_reference, x_true, reference_seconds)
    difference = torch.linalg.vector_norm(x - x_reference) / torch.linalg.vector_norm(x_reference)
    print(f"Relative difference from reference: {difference.item():.3e}")

torch.linalg.solve reference: 3.880423 s | relative residual 1.768e-15 | relative solution error 1.463e-12
Relative difference from reference: 1.310e-12


## Limits and interpretation

- bcsstk17 is 10,974 × 10,974: about 0.90 GiB per dense float64 matrix. A and LU together need about 1.8 GiB; loading and the optional reference need additional memory. This deliberately treats the sparse input as dense to demonstrate elimination.
- The rank-one update is `A22 ← A22 − multipliers ⊗ pivot_row`, performed in place with `addr_`. Only the two update vectors are cloned to avoid aliasing. Row swaps include the entire LU row and the transformed RHS. Backward substitution reads only U.
- This is an unblocked educational algorithm. GPU pivot decisions synchronize with Python at each step, and the rank-one updates repeatedly stream the trailing matrix. Vectorization does not guarantee a speedup over a CPU or a blocked library solver.
- FP64 performance depends on the assigned Kaggle GPU. T4 has limited FP64 throughput compared with its advertised lower-precision AI performance; a P100 is better suited to FP64 if available. Float64 is retained for accuracy. Expect bcsstk17 to take time; try bcsstk09 first if needed.
- Relative residual is `||Ax − b||₂ / ||b||₂`. Relative solution error is `||x − x_true||₂ / ||x_true||₂`, with `x_true = ones` and b formed in float64. A small residual does not guarantee a small solution error for an ill-conditioned matrix. SuiteSparse lists bcsstk17's condition number near 1.3e10.
- An exactly zero or nonfinite pivot raises an error, including the final pivot. No arbitrary near-zero tolerance is used; partial pivoting cannot remove ill-conditioning.

Sources: [HB/bcsstk17](https://sparse.tamu.edu/HB/bcsstk17), [HB/bcsstk09](https://sparse.tamu.edu/HB/bcsstk09), [PyTorch addr_](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.addr_.html), [CUDA synchronization](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html), [GPU precision comparison](https://docs.cloud.google.com/compute/docs/gpus/).
Kaggle setup: [Notebook documentation](https://www.kaggle.com/docs/notebooks), [NVIDIA Kaggle GPU setup](https://docs.nvidia.com/datascience/deployment/latest/platforms/kaggle/).